In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Math, Latex
import os
import subprocess
import shutil
import re

In [ ]:
def normsq3v(v):
    return v[0]**2 + v[1]**2 + v[2]**2

In [ ]:
# Check if the directory Quant_Files exists, otherwise create it

file_dir = "../run/Initial_Pos_Files/"
os.makedirs(file_dir, exist_ok=True)

add_randomness = 0.0



# Points generation on the surface of a sphere

Given a sphere of radius $R$ centered at the origin:

### Latitude $ (\phi)$: 
Determines the vertical placement, spaced using the inverse cosine function for uniformity.
$\left[
\phi = \arccos\left(1 - \frac{2i}{N - 1}\right)
\right]
$
where $i$ is the index of the point, and $N$ is the total number of points.

### Longitude $ (\theta)$: 
It is determined by the golden angle for even horizontal distribution.
$
\left[
\theta = 2\pi \cdot \text{golden\_angle\_ratio} \cdot i
\right]
$
where $\text{golden\_angle\_ratio} = \frac{1 + \sqrt{5}}{2}$.

### Cartesian Coordinates: 
$\begin{aligned}
x &= R \cdot \sin(\phi) \cdot \cos(\theta) \\
y &= R \cdot \sin(\phi) \cdot \sin(\theta) \\
z &= R \cdot \cos(\phi)
\end{aligned}
$

### Golden Section Spiral

This method ensures that the points are evenly distributed across the sphere without clustering.  
The golden angle, approximately $137.5^\circ$, ensures optimal spacing in the azimuthal direction.


In [ ]:
def save_points_to_file(points, file_name, radius, color):
    with open(file_name, "w") as file:
        for i, point in enumerate(points, start=1):
            file.write(f"{i} {point[0]:.2f} {point[1]:.2f} {point[2]:.2f} {radius} {color}\n")



In [ ]:
def visualize_points(inner_points, outer_points):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    inner_points = np.array(inner_points)
    outer_points = np.array(outer_points)

    ax.scatter(inner_points[:, 0], inner_points[:, 1], inner_points[:, 2], color='red', label='Inner Germ Cells')
    ax.scatter(outer_points[:, 0], outer_points[:, 1], outer_points[:, 2], color='blue', label='Outer Somatic Cells')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.legend()
    plt.show()

In [ ]:
# Function to calculate the radius of a sphere with a given volume
def sphere_radius(volume):
    return (3 * volume / (4 * np.pi)) ** (1/3)

# Function to calculate the semi-axes of an ellipsoid with a given volume
def ellipsoid_radii(volume, fixed_axis, b_to_c_ratio=1.0):
  
    a = fixed_axis
    c_squared = volume / ((4/3) * np.pi * a * b_to_c_ratio)
    c = np.sqrt(c_squared)
    b = b_to_c_ratio * c
    return (a, b, c)


# Function to calculate the radius and height of a cylinder with a given volume
def cylinder_dimensions(volume, height_to_radius_ratio):
    r = (volume / (np.pi * height_to_radius_ratio)) ** (1/3)
    h = height_to_radius_ratio * r
    return (r, h)

# Function to calculate dimensions for equal volume shapes
def calculate_equal_volume_shapes(sphere_radius, ellipsoid_fixed_axis, cylinder_height_to_radius_ratio, ellipsoid_b_to_c_ratio=1.0):
    sphere_volume = (4/3) * np.pi * sphere_radius**3
    ellipsoid_axes = ellipsoid_radii(sphere_volume, ellipsoid_fixed_axis, ellipsoid_b_to_c_ratio)
    cylinder_dims = cylinder_dimensions(sphere_volume, cylinder_height_to_radius_ratio)
    return {
        'sphere_radius': sphere_radius,
        'ellipsoid_axes': ellipsoid_axes,
        'cylinder_dimensions': cylinder_dims,
        'sphere_volume': sphere_volume
    }


In [ ]:
# Parameters across all shapes
sphere_radius = 9.5
ellipsoid_fixed_axis = 7.5  # Keep consistent across all cells
cylinder_height_to_radius_ratio = 3.5  # Keep consistent across all cells
ellipsoid_b_to_c_ratio = 0.5 # Ratio of b to c for ellipsoid
# Calculate dimensions for equal volume shapes
dimensions = calculate_equal_volume_shapes(sphere_radius, ellipsoid_fixed_axis, cylinder_height_to_radius_ratio, ellipsoid_b_to_c_ratio)


In [ ]:
def generate_points_on_sphere(num_points, radius=1, center=(0, 0, 0), domain=None, min_distance=0.1, randomness=0.01):
    points = []
    golden_angle = np.pi * (3 - np.sqrt(5))  # Golden angle in radians
    
    for i in range(num_points):
        z = 1 - (2 * i) / (num_points - 1)    # z-coordinate (mapped from -1 to 1)
        phi = np.arccos(z)                   # Latitude
        theta = golden_angle * i             # Longitude
        
        # Generate the point in spherical coordinates
        x = radius * np.sin(phi) * np.cos(theta) + center[0]
        y = radius * np.sin(phi) * np.sin(theta) + center[1]
        z = radius * z + center[2]
        
        # Add randomness to the coordinates
        x += np.random.uniform(-randomness, randomness)
        y += np.random.uniform(-randomness, randomness)
        z += np.random.uniform(-randomness, randomness)
        
        # Check if the point lies within the domain
        if domain:
            if not (domain['x'][0] <= x <= domain['x'][1] and
                    domain['y'][0] <= y <= domain['y'][1] and
                    domain['z'][0] <= z <= domain['z'][1]):
                continue
        
        # Ensure minimum distance between points
        if points:
            distances = np.linalg.norm(np.array(points) - np.array([x, y, z]), axis=1)
            if np.any(distances < min_distance):
                continue
        
        points.append((x, y, z))
    
    return np.array(points)

def add_spheroid(spheroid_params, worldsize, points, fill=True, randomness=0.01):
    c = np.array([float(spheroid_params[0]), float(spheroid_params[1]), float(spheroid_params[2])])
    spheroid_radius = float(spheroid_params[3])
    n = int(spheroid_params[4])

    if fill:
        i = 0
        while i < n:
            v = np.random.uniform(-1, 1, 3)
            if normsq3v(v) > 1:
                continue

            p = c + spheroid_radius * v

            # Add randomness to the coordinates
            p += np.random.uniform(-randomness, randomness, 3)

            # Fail conditions
            if (p[0] < 0.0 or p[0] > worldsize[0] or
                p[1] < 0.0 or p[1] > worldsize[1] or
                p[2] < 0.0 or p[2] > worldsize[2]):
                continue

            # Pass: add new point
            points.append(p)
            i += 1
    else:
        domain = {'x': (0, worldsize[0]), 'y': (0, worldsize[1]), 'z': (0, worldsize[2])}
        new_points = generate_points_on_sphere(n, radius=spheroid_radius, center=c, domain=domain, randomness=randomness)
        for p in new_points:
            # Pass: add new point
            points.append(p)


In [ ]:
# =================================================================
#                    SPHERE 
# =================================================================
worldsize = [40.0, 40.0, 40.0]
inner_germ_cells = []
outer_somatic_cells = []

offset = 2.0  # Offset for somatic cell layer
# Use calculated dimensions as OUTER boundary
outer_radius = dimensions['sphere_radius']  # noooo  +2 here
inner_radius = max(0.5, outer_radius - offset)   # Subtract offset for inner

# Fill the sphere with germ cells (inner)
spheroid_params_fill = ['20', '20', '20', str(inner_radius), '32']
add_spheroid(spheroid_params_fill, worldsize, inner_germ_cells, fill=True, randomness=0.01)

# Cover the sphere with somatic cells (outer - equivalent volume)
spheroid_params_cover = ['20', '20', '20', str(outer_radius), '128']
add_spheroid(spheroid_params_cover, worldsize, outer_somatic_cells, fill=False, randomness=0.01)

# Save points to files
input_shape = "sphere"  # Example shape
germ_file_name = f"{file_dir}/Germ_coordinates_{input_shape}.txt"
somatic_file_name = f"{file_dir}/Somatic_coordinates_{input_shape}.txt"

save_points_to_file(inner_germ_cells, germ_file_name, 40.0, 40)
save_points_to_file(outer_somatic_cells, somatic_file_name, 10.0, 10)

print("Germ (inner) and Somatic (outer) cell centers saved to separate files.")

# Print calculated dimensions for verification
print(f"Calculated sphere radius: {dimensions['sphere_radius']:.2f}")
print(f"Sphere volume: {dimensions['sphere_volume']:.2f}")


visualize_points(inner_germ_cells, outer_somatic_cells)


In [ ]:
# Function to generate points inside an ellipsoid (for filling)
def generate_points_inside_ellipsoid(num_points, radii, center=(0, 0, 0), randomness=0.01):
    points = []
    for _ in range(num_points):
        while True:
            # Generate random point in unit sphere
            x = np.random.uniform(-1, 1)
            y = np.random.uniform(-1, 1)
            z = np.random.uniform(-1, 1)
            
            # Check if point is inside unit sphere
            if x**2 + y**2 + z**2 <= 1:
                # Scale by ellipsoid radii and translate to center
                point_x = radii[0] * x + center[0]
                point_y = radii[1] * y + center[1]
                point_z = radii[2] * z + center[2]
                
                # Add randomness
                point_x += np.random.uniform(-randomness, randomness)
                point_y += np.random.uniform(-randomness, randomness)
                point_z += np.random.uniform(-randomness, randomness)
                
                points.append((point_x, point_y, point_z))
                break
    return points

def generate_points_on_ellipsoid(num_points, radii, center=(0, 0, 0), randomness=0.01):
    points = []
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))  # Golden angle in radians

    for i in range(num_points):
        z = 1 - (2 * i) / (num_points - 1)  # z-coordinate (mapped from -1 to 1)
        phi = np.arccos(z)  # Latitude
        theta = i * golden_angle  # Longitude

        # Generate the point in spherical coordinates
        x = radii[0] * np.sin(phi) * np.cos(theta) + center[0]
        y = radii[1] * np.sin(phi) * np.sin(theta) + center[1]
        z = radii[2] * z + center[2]

        # Add randomness to the coordinates
        x += np.random.uniform(-randomness, randomness)
        y += np.random.uniform(-randomness, randomness)
        z += np.random.uniform(-randomness, randomness)

        points.append((x, y, z))

    return points


In [ ]:

# =================================================================
#                    ELLIPSOID 
# =================================================================
worldsize = [40.0, 40.0, 40.0]
inner_germ_cells = []
outer_somatic_cells = []


# Use the calculated dimensions for the OUTER ellipsoid (equivalent volume)
ellipsoid_axes_cover = dimensions['ellipsoid_axes']  # This has equivalent volume to sphere

# Calculate inner ellipsoid dimensions (smaller than outer)
offset = 2  # Thickness of the somatic layer
ellipsoid_axes_fill = (
    max(0.5, ellipsoid_axes_cover[0] - offset),  # Ensure positive values
    max(0.5, ellipsoid_axes_cover[1] - offset),
    max(0.5, ellipsoid_axes_cover[2] - offset)
)

# Print calculated dimensions for verification
print(f"Calculated ellipsoid axes (outer/cover): {ellipsoid_axes_cover}")
print(f"Calculated ellipsoid axes (inner/fill): {ellipsoid_axes_fill}")
print(f"Sphere volume: {dimensions['sphere_volume']:.2f}")
print(f"Outer ellipsoid volume: {(4/3) * np.pi * ellipsoid_axes_cover[0] * ellipsoid_axes_cover[1] * ellipsoid_axes_cover[2]:.2f}")
print(f"Inner ellipsoid volume: {(4/3) * np.pi * ellipsoid_axes_fill[0] * ellipsoid_axes_fill[1] * ellipsoid_axes_fill[2]:.2f}")

# Fill the ellipsoid with germ cells using calculated dimensions (inner)
center_fill = (20.0, 20.0, 20.0)  # Center coordinates
num_points_fill = 32
inner_germ_cells = generate_points_inside_ellipsoid(num_points_fill, ellipsoid_axes_fill, center=center_fill, randomness=0.01)

# Cover the ellipsoid with somatic cells using calculated dimensions (outer - equivalent volume)
center_cover = (20.0, 20.0, 20.0)  # Same center
num_points_cover = 128
outer_somatic_cells = generate_points_on_ellipsoid(num_points_cover, ellipsoid_axes_cover, center=center_cover, randomness=0.01)

# Save points to files
input_shape = "ellipsoid"
germ_file_name = f"{file_dir}/Germ_coordinates_{input_shape}.txt"
somatic_file_name = f"{file_dir}/Somatic_coordinates_{input_shape}.txt"

save_points_to_file(inner_germ_cells, germ_file_name, 40.0, 40)
save_points_to_file(outer_somatic_cells, somatic_file_name, 10, 10)

print("Germ (inner) and Somatic (outer) cell centers saved to separate files.")

visualize_points(inner_germ_cells, outer_somatic_cells)

In [ ]:
def generate_points_inside_cylinder(num_points, height, radius, center=(0, 0, 0), randomness=0.01):
    points = []
    for _ in range(num_points):
        while True:
            z = np.random.uniform(-height / 2, height / 2) + center[2]
            theta = np.random.uniform(0, 2 * np.pi)
            r = radius * np.sqrt(np.random.uniform(0, 1))
            x = r * np.cos(theta) + center[0]
            y = r * np.sin(theta) + center[1]

            # Add randomness to the coordinates
            x += np.random.uniform(-randomness, randomness)
            y += np.random.uniform(-randomness, randomness)
            z += np.random.uniform(-randomness, randomness)

            points.append((x, y, z))
            break
    return points

def generate_points_on_cylinder(num_points, height, radius, center=(0, 0, 0), randomness=0.01):
    points = []
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))  # Golden angle in radians

    for i in range(num_points):
        z = height * (i / float(num_points - 1)) - height / 2.0 + center[2]  # Height position
        theta = i * golden_angle  # Angle around the circumference
        x = radius * np.cos(theta) + center[0]
        y = radius * np.sin(theta) + center[1]

        # Add randomness to the coordinates
        x += np.random.uniform(-randomness, randomness)
        y += np.random.uniform(-randomness, randomness)
        z += np.random.uniform(-randomness, randomness)

        points.append((x, y, z))

    return points

def generate_points_on_cylinder_caps(num_points, radius, center=(0, 0, 0), height=0, randomness=0.01):
    points = []
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))  # Golden angle in radians

    for i in range(num_points):
        theta = i * golden_angle  # Angle around the circumference
        x = radius * np.cos(theta) + center[0]
        y = radius * np.sin(theta) + center[1]
        z_top = height / 2.0 + center[2]
        z_bottom = -height / 2.0 + center[2]

        # Add randomness to the coordinates
        x += np.random.uniform(-randomness, randomness)
        y += np.random.uniform(-randomness, randomness)

        points.append((x, y, z_top))
        points.append((x, y, z_bottom))

    return points

def generate_points_on_circular_plate(num_points, radius, z, center=(0, 0, 0), randomness=0.01):
    points = []
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))  # Golden angle in radians

    for i in range(num_points):
        r = radius * np.sqrt(i / float(num_points - 1))
        theta = i * golden_angle
        x = r * np.cos(theta) + center[0]
        y = r * np.sin(theta) + center[1]

        # Add randomness to the coordinates
        x += np.random.uniform(-randomness, randomness)
        y += np.random.uniform(-randomness, randomness)

        points.append((x, y, z))

    return points

In [ ]:
# =================================================================
#                    CYLINDER
# =================================================================
worldsize = [40.0, 40.0, 40.0]
inner_germ_cells = []
outer_somatic_cells = []

# Extract calculated cylinder dimensions - USE THESE AS OUTER DIMENSIONS
outer_radius, outer_height = dimensions['cylinder_dimensions']

# Calculate inner dimensions by subtracting offset
layer_offset = 2.0
inner_radius = max(0.5, outer_radius - layer_offset)
inner_height = max(0.5, outer_height - layer_offset)

print(f"Outer cylinder: radius={outer_radius:.2f}, height={outer_height:.2f}")
print(f"Inner cylinder: radius={inner_radius:.2f}, height={inner_height:.2f}")

# Fill the cylinder with germ cells (using inner dimensions)
center = (20.0, 20.0, 20.0)
num_points_fill = 32
inner_germ_cells = generate_points_inside_cylinder(num_points_fill, inner_height, inner_radius, center=center, randomness=add_randomness)

# Cover the cylinder with somatic cells (using outer dimensions)
num_points_cover = 128

# FIXED: Better point distribution
num_points_surface = int(num_points_cover * 0.7)  # 65% on curved surface (83 points)
num_points_caps = int(num_points_cover * 0.02)     # 5% on cap edges (6 points)
num_points_plates = num_points_cover - num_points_surface - num_points_caps * 2  # 30% on plates (39 points)

print(f"Point distribution: Surface={num_points_surface}, Cap edges={num_points_caps*2}, Plates={num_points_plates}")

# Generate points with proper connectivity
outer_somatic_cells = []

# 1. Curved surface points (connect smoothly)
surface_points = generate_points_on_cylinder(num_points_surface, outer_height, outer_radius, center=center, randomness=add_randomness)
outer_somatic_cells.extend(surface_points)

# 2. Cap edge points (at exact top and bottom of cylinder)
if num_points_caps > 0:
    cap_points = generate_points_on_cylinder_caps(num_points_caps, outer_radius, center=center, height=outer_height, randomness=add_randomness)
    outer_somatic_cells.extend(cap_points)

# 3. Circular plates (fill the caps)
if num_points_plates > 0:
    # Top plate
    top_plate_points = generate_points_on_circular_plate(
        num_points_plates // 2, 
        outer_radius, 
        center[2] + outer_height / 2,  # Top Z position
        center=center, 
        randomness=add_randomness
    )
    outer_somatic_cells.extend(top_plate_points)
    
    # Bottom plate  
    bottom_plate_points = generate_points_on_circular_plate(
        num_points_plates - num_points_plates // 2,  # Handle odd numbers
        outer_radius,
        center[2] - outer_height / 2,  # Bottom Z position
        center=center,
        randomness=add_randomness
    )
    outer_somatic_cells.extend(bottom_plate_points)

# VERIFICATION: Check Z-coordinate ranges
if outer_somatic_cells:
    z_coords = [point[2] for point in outer_somatic_cells]
    print(f"\nZ-coordinate verification:")
    print(f"  Min Z: {min(z_coords):.2f} (should be ~{center[2] - outer_height/2:.2f})")
    print(f"  Max Z: {max(z_coords):.2f} (should be ~{center[2] + outer_height/2:.2f})")
    print(f"  Center Z: {center[2]:.2f}")
    print(f"  Expected range: [{center[2] - outer_height/2:.2f}, {center[2] + outer_height/2:.2f}]")

# Save points to files
input_shape = "cylinder"
germ_file_name = f"{file_dir}/Germ_coordinates_{input_shape}.txt"
somatic_file_name = f"{file_dir}/Somatic_coordinates_{input_shape}.txt"

save_points_to_file(inner_germ_cells, germ_file_name, 40.0, 40)
save_points_to_file(outer_somatic_cells, somatic_file_name, 10, 10)

print("Germ (inner) and Somatic (outer) cell centers saved to separate files.")

# Statistics
radius_fill = inner_radius
height_fill = inner_height
radius_cover = outer_radius
height_cover = outer_height

def cylinder_volume(radius, height):
    return np.pi * radius**2 * height

print(f"\n=== FINAL CYLINDER STATISTICS ===")
print(f"Inner cylinder: radius={radius_fill:.2f}, height={height_fill:.2f}, volume={cylinder_volume(radius_fill, height_fill):.2f}")
print(f"Outer cylinder: radius={radius_cover:.2f}, height={height_cover:.2f}, volume={cylinder_volume(radius_cover, height_cover):.2f}")
print(f"Volume ratio (outer/inner): {cylinder_volume(radius_cover, height_cover) / cylinder_volume(radius_fill, height_fill):.2f}")
print(f"Total outer points: {len(outer_somatic_cells)}")

visualize_points(inner_germ_cells, outer_somatic_cells)

In [ ]:
# Add volume verification function
def verify_equivalent_volumes(dimensions):
    sphere_vol = (4/3) * np.pi * dimensions['sphere_radius']**3
    ellipsoid_vol = (4/3) * np.pi * dimensions['ellipsoid_axes'][0] * dimensions['ellipsoid_axes'][1] * dimensions['ellipsoid_axes'][2]
    cylinder_vol = np.pi * dimensions['cylinder_dimensions'][0]**2 * dimensions['cylinder_dimensions'][1]
    
    print(f"=== VOLUME VERIFICATION ===")
    print(f"Sphere volume: {sphere_vol:.2f}")
    print(f"Ellipsoid volume: {ellipsoid_vol:.2f}")
    print(f"Cylinder volume: {cylinder_vol:.2f}")
    print(f"Volume differences:")
    print(f"  Ellipsoid vs Sphere: {abs(ellipsoid_vol - sphere_vol):.2f}")
    print(f"  Cylinder vs Sphere: {abs(cylinder_vol - sphere_vol):.2f}")
    return sphere_vol, ellipsoid_vol, cylinder_vol

# Call after calculating dimensions
verify_equivalent_volumes(dimensions)